# 科创潜在客户数据预处理 & 探索

**数据来源**：科创人才表 ∩ 消贷账户表（2025-12-31 快照），拼接授信台账、客户基本信息、手机银行统计、消费偏好

**因变量**：
- `y_dq_risk`：rt_acct_stat_2 ∈ {3,4,7,9} → 1（不良）
- `y_freq`：ac_curr_bal_diff 与 ba_out_bal_diff 同时处于前20% → 1（高活跃）

**采样与阈值策略**：
- 采样时统一尽量调整到正样本:负样本 = 1:1
- 模型选择验证集只用于候选方法的 AUC / Precision 选择，不再参与 LightGBM 早停
- 按技术文档，最终测试采用测试集 Top-ρ 分位数阈值；概率校准集留给任务3训练概率校准器

---

## Cell 1：参数配置（在这里修改路径和参数）

In [ ]:
# ── 文件路径 ──
FILE_PATH   = "kechuang_merged0729.csv"   # 第一份数据（Excel 或 CSV）
FILE_PATH_2 = ""      # 第二份数据（Excel 或 CSV，留空字符串则跳过重叠分析）

# ── 快照日期（用于 days_since_become_cust 等衍生特征） ──
SNAPSHOT_DATE = "2026-06-24"

# ── 建模目标 ──
# 【任务3前置要求】本 Notebook 必须完整运行两次：TARGET 分别设为 y_freq 和 y_dq_risk。
# 两次都要从 Cell 1 运行到最后，并保留 sampling_method_selection_y_freq.csv
# 与 sampling_method_selection_y_dq_risk.csv；任务3需要分别读取两者 selected=True 的结果。
# "y_freq"    : 频率活跃度标签（贷款周期内余额/支用变化）
# "y_dq_risk" : 违约风险标签（到期日账户状态为3/4/7/9 → 1）
TARGET = "y_freq"

# ── y_freq 因变量构造模式 ──
# "bout_gt0_and_curr_p80" : ba_out_bal_diff > 0  且  ac_curr_bal_diff >= P80（默认）
# "bout_p80_and_accr_p80" : ba_out_bal_diff >= P80  且  ac_accr_bal_diff >= P80
# "curr_p80_only"         : ac_curr_bal_diff >= P80（单条件，仅看余额差值前20%）
# "curr_p80_and_bout_p80" : ac_curr_bal_diff >= P80  且  ba_out_bal_diff >= P80（两者均前20%）
Y_FREQ_MODE = "curr_p80_and_bout_p80"  # 仅 TARGET="y_freq" 时有效

# ── 授信额度多项式特征 ──
ADD_QUOTA_SQ    = False
ADD_QUOTA_CUBE  = False
ADD_QUOTA_LOG   = False

# ── 到期日筛选开关（对两份数据均生效） ──
APPLY_MATURITY_FILTER = False
MATURITY_CUTOFF       = "2026-07-21"   # 两份数据统一用同一个到期日阈值

# ── 贷款生效日筛选：当前开启，仅保留以下日期区间内生效的贷款 ──
APPLY_EFF_DATE_FILTER = False
EFF_DATE_LOWER        = "2025-01-01"
EFF_DATE_UPPER        = "2026-03-31"

# ── LightGBM 建模参数 ──
RANDOM_STATE          = 42
TEST_SIZE             = 0.2   # 旧参数保留兼容；实际按照以下4个比例划分数据集
TRAIN_RATIO          = 0.60
VALIDATION_RATIO     = 0.15
CALIBRATION_RATIO    = 0.15
FINAL_TEST_RATIO     = 0.10
INNER_EARLY_STOP_RATIO = 0.20  # 仅在60%训练集内部用于确定LightGBM迭代轮数
# 候选方法在验证集确定；最终测试按技术文档使用测试集正样本率对应的 Top-ρ 分位数阈值。
# 概率校准集不参与任务2的模型训练或方法选择，留给任务3训练概率校准器。

# ── 贷款账户重复处理：True 仅删除完全相同副本，同键冲突会报错；False 仅输出统计 ──
DEDUP_CST_LOAN         = False
# 与额度优化统一客户池：两个目标都剔除起点已违约客户，确保任务2选型可直接用于任务3
EXCLUDE_DQ_START_CUSTOMERS = True
HANDLE_IMBALANCE      = False    # 旧参数保留供原 SHAP 后备训练；候选 baseline 不使用类别权重
EARLY_STOPPING_ROUNDS = 50
LGB_PARAMS = {
    "objective":        "binary",
    "metric":           "auc",
    "n_estimators":     500,
    "learning_rate":    0.05,
    "num_leaves":       31,
    "max_depth":        -1,
    "min_child_samples": 20,
    "subsample":        0.8,
    "bagging_freq":     1,   # subsample 只有在 bagging_freq > 0 时才生效
    "colsample_bytree": 0.8,
    "reg_alpha":        0.1,
    "reg_lambda":       0.1,
    "random_state":     42,
    "verbose":          -1,
}

# ════════════════════════════════════════════════════════
# 采样参数配置
# ════════════════════════════════════════════════════════
# SAMPLING_METHODS：填入想对比的方法列表，留空列表则只跑无采样基线。
# 采样只作用于训练集；验证、校准和测试集始终保持各自时间段的原始分布。
#   过采样: 'smote' / 'borderline_smote' / 'adasyn'
#   欠采样: 'random_under' / 'tomek' / 'enn'
#   组合:   'smoteenn' / 'smotetomek'
#   集成:   'balance_cascade'（分类器驱动级联）/ 'easy_ensemble'（独立随机欠采样集成）
SAMPLING_METHODS = ["random_over", "smote", "borderline_smote",'adasyn','smoteenn','smotetomek', "random_under", "balance_cascade", "easy_ensemble"]  # ← 手动控制

# 普通采样方法目标 正样本数/负样本数；1.0 = 训练集采样后正负 1:1
SAMPLING_STRATEGY = 1.0
# BalanceCascade / Ensemble 子模型数量，以及每个子集中 负样本数/正样本数；1.0 = 每个子集正负 1:1
SAMPLING_N_ESTIMATORS = 10
SAMPLING_ENSEMBLE_RATIO = 1.0

# 名义类别字段的显式候选；age 是连续变量，不得放入该列表。
# SMOTE/SMOTEENN/SMOTETomek 按 SMOTENC 规则处理；BorderlineSMOTE/ADASYN 是混合特征扩展。
SAMPLING_CATEGORICAL_CANDIDATES = [
    "gnd_cd", "mar_sttn_cd", "education_cd", "occup_cd", "cst_star_cd",
    "busikind",
]

# SHAP 参数
SHAP_SAMPLE_SIZE = 500     # SHAP 计算时随机抽取的样本数（样本量大时控制速度）
SHAP_TOP_N       = 20      # SHAP 图展示 Top N 特征


# ════════════════════════════════════════════════════════
# 模型配置阶段：三个互斥运行模式
# ════════════════════════════════════════════════════════
# 1) True / False：完整比较所有 SAMPLING_METHODS，并保存最优方法、超参数和最佳轮数。
# 2) False / True：方法已经确定，只在模型训练集内部对该方法早停，更新最佳轮数并保存。
# 3) False / False：完全复用 SELECTED_CONFIG_PATH 中的历史配置。
# 两个开关不能同时为 True。
RUN_MODEL_SELECTION = False
REFIT_BEST_ROUNDS_ONLY = True

# 当尚无历史配置文件时使用的方法兜底；正常复用时会被保存配置覆盖。
FORCED_METHOD_KEY = "easy_ensemble"
FORCED_METHOD_LABEL = "Easy Ensemble（随机欠采样集成）"
# 仅作为“历史配置文件尚不存在”时的兜底。已有配置后不需要手工维护。
# 建议首次先运行“完整方法比较”或“只更新最佳轮数”；也可填入可信历史轮数。
FORCED_ROUNDS = None

# 独立保存“方法选择阶段”的结果，供全量模型训练复用。
SELECTED_CONFIG_PATH = f"selected_model_config_{TARGET}.json"

# 全量 SHAP 模型阶段：
# "auto"  = 配置未更新且完整模型制品存在时加载，否则按选定配置全量训练并保存；
# "train" = 强制按选定配置在全量数据上重训并覆盖制品；
# "load"  = 只加载已有全量模型，缺失时报错（不能与上面两个更新配置模式同时使用）。
FULL_MODEL_ACTION = "auto"
ARTIFACT_ROOT = f"full_shap_model_{TARGET}"
MODEL_DIR = ARTIFACT_ROOT
PREPROCESSOR_PATH = f"model_preprocessor_{TARGET}.pkl"
METADATA_PATH = f"full_shap_model_metadata_{TARGET}.json"


## Cell 2：导入模块

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# 中文字体（离线机器可换为 'SimHei' 或 'Source Han Sans CN'）
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 预处理模块（与本 notebook 放在同一目录）
from load_kechuang_potential_data import (
    load_kechuang_potential, get_feature_stats,
    get_feature_descriptive_stats, get_consumption_field_cn_map,
    PotentialFeaturePreprocessor, stratified_dual_target_partition_indices,
)

print("模块导入完成")

## Cell 3：运行预处理

In [ ]:
(df_clean, feature_missing_df, label_stats_df, X, y, feature_names, thresholds,
 field_coverage_summary_df, field_coverage_monthly_df) = load_kechuang_potential(
    file_path              = FILE_PATH,
    snapshot_date          = SNAPSHOT_DATE,
    target                 = TARGET,
    add_quota_sq           = ADD_QUOTA_SQ,
    add_quota_cube         = ADD_QUOTA_CUBE,
    add_quota_log          = ADD_QUOTA_LOG,
    apply_maturity_filter  = APPLY_MATURITY_FILTER,
    maturity_cutoff        = MATURITY_CUTOFF,
    y_freq_mode            = Y_FREQ_MODE,
    apply_eff_date_filter  = APPLY_EFF_DATE_FILTER,
    eff_date_lower         = EFF_DATE_LOWER,
    eff_date_upper         = EFF_DATE_UPPER,
    dedup_cst_loan         = DEDUP_CST_LOAN,
    exclude_dq_start_customers = EXCLUDE_DQ_START_CUSTOMERS,
    label_threshold_train_ratio = TRAIN_RATIO,
    require_dual_label_cohort = True,
)


## 字段覆盖与全部特征描述统计


In [ ]:
# 字段日期覆盖统计在读取后完成列名小写/科创字段改名时立即计算，
# 尚未经过贷款筛选、去重或一客多贷聚合。
print('字段覆盖汇总（稳定覆盖起始月：连续3个月覆盖率均达到90%的最早月份）')
display(field_coverage_summary_df)

print('字段按贷款生效月份覆盖明细')
display(field_coverage_monthly_df)

# 清洗结束后的全部原始特征，不包含因变量、客户ID和标签阈值参考期辅助列。
_desc_exclude = {'cst_id', 'y_freq', 'y_dq_risk', 'split_eff_date'}
_desc_feature_cols = [c for c in df_clean.columns if c not in _desc_exclude]
numeric_feature_desc_df, categorical_feature_desc_df = get_feature_descriptive_stats(
    df_clean, _desc_feature_cols
)

print('数值型特征描述统计（最小值、最大值、平均值）')
display(numeric_feature_desc_df)

print('类别型特征描述统计（众数及每个类别比例）')
display(categorical_feature_desc_df)

# ── 保存完整统计结果：Notebook页面可能截断，CSV文件保留全部行 ──
from pathlib import Path
STATS_OUTPUT_DIR = Path(f'字段覆盖与全部特征描述性统计结果_{TARGET}')
STATS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
_stats_tables = {
    '字段覆盖汇总.csv': field_coverage_summary_df,
    '字段月度覆盖明细.csv': field_coverage_monthly_df,
    '数值型特征描述统计.csv': numeric_feature_desc_df,
    '类别型特征描述统计.csv': categorical_feature_desc_df,
}
for _file_name, _table in _stats_tables.items():
    _table.to_csv(STATS_OUTPUT_DIR / _file_name, index=False, encoding='utf-8-sig')

print(f'\n✅ 完整统计结果已保存至文件夹: {STATS_OUTPUT_DIR.resolve()}')
print(f'  字段覆盖汇总      : {len(field_coverage_summary_df):,} 行')
print(f'  字段月度覆盖明细  : {len(field_coverage_monthly_df):,} 行')
print(f'  数值型特征描述统计: {len(numeric_feature_desc_df):,} 行')
print(f'  类别型特征描述统计: {len(categorical_feature_desc_df):,} 行')


## Cell 4：数据规模概览

In [ ]:
print("━" * 50)
print(f"  客户数（行数）     : {len(df_clean):,}")
print(f"  特征数（建模用）    : {len(feature_names):,}")
print(f"  清洗后总列数        : {df_clean.shape[1]:,}")
print("━" * 50)
print(f"  建模目标            : {TARGET}")
print(f"  到期日筛选          : {'开启，cutoff=' + MATURITY_CUTOFF if APPLY_MATURITY_FILTER else '关闭'}")
if TARGET == 'y_freq':
    print(f"  y_freq 构造模式     : {thresholds['y_freq_mode']}")
    if thresholds['y_freq_mode'] == 'bout_gt0_and_curr_p80':
        print(f"  y_freq 条件①       : ba_out_bal_diff > 0")
        print(f"  y_freq 条件②       : ac_curr_bal_diff >= {thresholds['thr_curr']:.4f}（P80）")
    elif thresholds['y_freq_mode'] == 'bout_p80_and_accr_p80':
        print(f"  y_freq 条件①       : ba_out_bal_diff  >= {thresholds['thr_bout']:.4f}（P80）")
        print(f"  y_freq 条件②       : ac_accr_bal_diff >= {thresholds['thr_accr']:.4f}（P80）")
    elif thresholds['y_freq_mode'] == 'curr_p80_only':
        print(f"  y_freq 条件         : ac_curr_bal_diff >= {thresholds['thr_curr']:.4f}（P80，单条件）")
    else:  # curr_p80_and_bout_p80
        print(f"  y_freq 条件①       : ac_curr_bal_diff >= {thresholds['thr_curr']:.4f}（P80）")
        print(f"  y_freq 条件②       : ba_out_bal_diff  >= {thresholds['thr_bout']:.4f}（P80）")
    print(f"  正样本率            : {y.mean():.4%}  ({y.sum()} / {len(y)})")
else:  # y_dq_risk
    print(f"  y_dq_risk 构造规则  : rt_acct_stat_2_end ∈ {{3,4,7,9}} → 1")
    print(f"  正样本率            : {y.mean():.4%}  ({y.sum()} / {len(y)})")
print("━" * 50)


## Cell 5：因变量统计

In [ ]:
print(f"因变量统计（{TARGET}）：")
display(label_stats_df)


## Cell 6：特征缺失率总览

In [ ]:
print(f"共 {len(feature_missing_df)} 个特征")
print("\n── 缺失率 > 0 的特征 ──")
nonzero_miss = feature_missing_df[feature_missing_df['missing_rate'] > 0]
if len(nonzero_miss) == 0:
    print("  所有特征缺失率均为 0（已填充完毕）")
else:
    display(nonzero_miss.reset_index(drop=True))

print("\n── 缺失率为 0 的特征数 ──")
print(f"  {(feature_missing_df['missing_rate'] == 0).sum()} 个")

## Cell 7：特征缺失率可视化（Top 30）

In [ ]:
top_miss = feature_missing_df[feature_missing_df['missing_rate'] > 0].head(30)

if len(top_miss) == 0:
    print("所有特征缺失率均为 0，无需绘图")
else:
    fig, ax = plt.subplots(figsize=(10, max(4, len(top_miss) * 0.35)))
    bars = ax.barh(
        top_miss['feature'][::-1],
        top_miss['missing_rate'][::-1] * 100,
        color='steelblue', edgecolor='white'
    )
    ax.set_xlabel('缺失率 (%)')
    ax.set_title('特征缺失率 Top 30')
    ax.axvline(x=40, color='red', linestyle='--', linewidth=1, label='40% 参考线')
    ax.legend()
    for bar, val in zip(bars, top_miss['missing_rate'][::-1]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val*100:.1f}%', va='center', fontsize=8)
    plt.tight_layout()
    plt.show()

## Cell 8：完整特征列表（全部打印）

In [ ]:
print(f"建模特征共 {len(feature_names)} 个：\n")
for i, name in enumerate(feature_names, 1):
    miss_row = feature_missing_df[feature_missing_df['feature'] == name]
    miss_str = miss_row['missing_rate_pct'].values[0] if len(miss_row) > 0 else "0.00%"
    print(f"  {i:3d}. {name:<40s}  缺失率: {miss_str}")

## Cell 9：清洗后数据预览

In [ ]:
display(df_clean.head(5))

## Cell 10：ba_out_bal_diff 数值分布统计

In [ ]:
# ── ba_out_bal_diff 数值分布统计 ──
# 依赖：Cell 3 已运行，df_clean 中存在 ba_out_bal_diff 列（因变量构造前保留，Part2 清洗后已删除）
# 注意：若 df_clean 中已无此列，请改用 Cell 3 运行前的中间结果，或在 build_labels_potential 输出中查看打印统计

import numpy as np

_col = "ba_out_bal_diff"

if _col not in df_clean.columns:
    print(f"⚠️  df_clean 中不存在 {_col}（已在 Part2 清洗中删除）")
    print("   请参考 Cell 3 运行输出中的'关键字段分布统计'部分查看分位数信息")
else:
    _s = df_clean[_col].dropna()
    _n = len(df_clean)

    print("=" * 52)
    print(f"【{_col} 基本描述统计】")
    print(df_clean[_col].describe(percentiles=[.05, .1, .25, .5, .75, .9, .95, .99]))

    print("\n" + "=" * 52)
    print("【特殊值计数】")
    _null = df_clean[_col].isnull().sum()
    _zero = (_s == 0).sum()
    _pos  = (_s >  0).sum()
    _neg  = (_s <  0).sum()
    print(f"  总行数        : {_n:,}")
    print(f"  缺失值 (NaN)  : {_null:,}  ({_null/_n:.2%})")
    print(f"  等于 0        : {_zero:,}  ({_zero/_n:.2%})")
    print(f"  大于 0 (支用) : {_pos:,}  ({_pos/_n:.2%})")
    print(f"  小于 0 (还款) : {_neg:,}  ({_neg/_n:.2%})")

    print("\n" + "=" * 52)
    print("【分桶分布】")
    _raw_bins = [_s.min(), -10000, -1000, -100, 0, 100, 1000, 10000, _s.max()]
    _bins = sorted(set(_raw_bins))
    try:
        _cut = pd.cut(_s, bins=_bins, include_lowest=True, right=True)
        _dist = _cut.value_counts(sort=False).reset_index()
        _dist.columns = ["区间", "计数"]
        _dist["占比"] = (_dist["计数"] / _n * 100).round(2).astype(str) + "%"
        print(_dist.to_string(index=False))
    except Exception as _e:
        print(f"  分桶失败（数据范围可能过窄）: {_e}")

    print("\n" + "=" * 52)
    print("【非零样本分位数】")
    _nz = _s[_s != 0]
    if len(_nz) > 0:
        for _p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
            print(f"  P{_p:>2}  : {np.percentile(_nz, _p):>15,.2f}")
    else:
        print("  无非零样本")

## Cell 11：保存清洗后数据

In [ ]:
# 保存完整清洗数据（含 cst_id、y 列、所有特征）
OUTPUT_PATH = f"kechuang_potential_cleaned_{TARGET}.xlsx"
df_clean.to_excel(OUTPUT_PATH, index=False)
print(f"已保存清洗后数据至: {OUTPUT_PATH}")
print(f"  行数: {len(df_clean):,}，列数: {df_clean.shape[1]:,}")

## Cell 12：划分 + 导入

In [ ]:
# ── Cell 12：双标签联合分层随机划分 + 训练集内部早停划分 + 导入 ──

import lightgbm as lgb
import numpy as np
import pandas as pd
import shap
import json
import os
import joblib
from sklearn.metrics import (
    roc_auc_score, accuracy_score, recall_score, precision_score,
    f1_score, confusion_matrix, roc_curve,
    precision_recall_curve, average_precision_score, matthews_corrcoef
)
from sampling_methods import BalanceCascade, sampler_factory, print_sampling_summary
# RobustScaler 只拟合采样器收到的训练数据，仅用于近邻距离；原始特征尺度仍供 LightGBM 使用。

OVERALL_POS_RATE = float(y.mean())
if not np.isclose(
    TRAIN_RATIO + VALIDATION_RATIO + CALIBRATION_RATIO + FINAL_TEST_RATIO, 1.0
):
    raise ValueError("四段切分比例之和必须等于 1。")

if not (len(df_clean) == len(X) == len(y)):
    raise AssertionError("df_clean、X、y 行数不一致，无法按同一位置切分。")

# 与额度优化任务一致：按 y_freq × y_dq_risk 联合标签分层随机切成 60% / 15% / 15% / 10%。
# 因为两个 TARGET 共用同一次联合分层，分别运行时会得到完全相同的客户划分。
# y_freq 的数据驱动阈值仍在加载阶段仅用最早 TRAIN_RATIO 客户拟合；它不是模型训练集。
train_pos, val_pos, cal_pos, test_pos = stratified_dual_target_partition_indices(
    df_clean,
    (TRAIN_RATIO, VALIDATION_RATIO, CALIBRATION_RATIO, FINAL_TEST_RATIO),
    random_state=RANDOM_STATE,
)
_row_index = df_clean.index.to_numpy()
train_rows = pd.Series(_row_index[train_pos], name='row_id')
val_rows = pd.Series(_row_index[val_pos], name='row_id')
cal_rows = pd.Series(_row_index[cal_pos], name='row_id')
test_rows = pd.Series(_row_index[test_pos], name='row_id')
y_train = y.loc[train_rows.to_numpy()].copy()
y_val = y.loc[val_rows.to_numpy()].copy()
y_cal = y.loc[cal_rows.to_numpy()].copy()
y_test = y.loc[test_rows.to_numpy()].copy()

# 快速路径优先复用与全量模型配套保存的预处理器；首次运行或强制训练时重新拟合。
_model_files_exist = os.path.isdir(MODEL_DIR) and any(
    name.startswith("model_") and name.endswith(".txt") for name in os.listdir(MODEL_DIR)
)
_artifacts_ready = (
    os.path.isfile(PREPROCESSOR_PATH)
    and os.path.isfile(METADATA_PATH)
    and _model_files_exist
)
if RUN_MODEL_SELECTION and REFIT_BEST_ROUNDS_ONLY:
    raise ValueError("RUN_MODEL_SELECTION 与 REFIT_BEST_ROUNDS_ONLY 不能同时为 True。")
if FULL_MODEL_ACTION not in {"auto", "train", "load"}:
    raise ValueError("FULL_MODEL_ACTION 只能是 'auto'、'train' 或 'load'。")
if FULL_MODEL_ACTION == "load" and (RUN_MODEL_SELECTION or REFIT_BEST_ROUNDS_ONLY):
    raise ValueError("更新方法/轮数后必须重训全量模型；请将 FULL_MODEL_ACTION 改为 'auto' 或 'train'。")
if FULL_MODEL_ACTION == "load" and not _artifacts_ready:
    raise FileNotFoundError("load 模式要求模型目录、预处理器和元数据全部存在。")
_configuration_updated = RUN_MODEL_SELECTION or REFIT_BEST_ROUNDS_ONLY
_load_saved_artifacts = FULL_MODEL_ACTION == "load" or (
    FULL_MODEL_ACTION == "auto" and _artifacts_ready and not _configuration_updated
)

if _load_saved_artifacts:
    with open(METADATA_PATH, "r", encoding="utf-8") as f:
        _saved_metadata = json.load(f)
    if _saved_metadata.get("target") != TARGET:
        raise ValueError("保存制品的 TARGET 与当前 TARGET 不一致。")
    model_preprocessor = joblib.load(PREPROCESSOR_PATH)
    print(f"✅ 已加载预处理器：{PREPROCESSOR_PATH}")
else:
    model_preprocessor = PotentialFeaturePreprocessor(
        target=TARGET,
        add_quota_sq=ADD_QUOTA_SQ,
        add_quota_cube=ADD_QUOTA_CUBE,
        add_quota_log=ADD_QUOTA_LOG,
        categorical_features=SAMPLING_CATEGORICAL_CANDIDATES,
    ).fit(df_clean.loc[train_rows.to_numpy()])
    print("✅ 已在训练集拟合新的预处理器")
X_train = model_preprocessor.transform(df_clean.loc[train_rows.to_numpy()])
X_val = model_preprocessor.transform(df_clean.loc[val_rows.to_numpy()])
X_cal = model_preprocessor.transform(df_clean.loc[cal_rows.to_numpy()])
X_test = model_preprocessor.transform(df_clean.loc[test_rows.to_numpy()])
X = model_preprocessor.transform(df_clean)
feature_names = list(model_preprocessor.feature_names_)
label_encoders = model_preprocessor.label_encoders_

# 只在60%训练集内部再划分实际拟合子集和早停集。
# 15%模型选择验证集不再参与LightGBM早停，只用于候选采样方法比较。
if not 0 < INNER_EARLY_STOP_RATIO < 1:
    raise ValueError("INNER_EARLY_STOP_RATIO 必须在 (0, 1) 内。")
inner_train_pos, early_stop_pos = stratified_dual_target_partition_indices(
    df_clean.loc[train_rows.to_numpy()],
    (1.0 - INNER_EARLY_STOP_RATIO, INNER_EARLY_STOP_RATIO),
    random_state=RANDOM_STATE + 1,
)
train_row_array = train_rows.to_numpy()
inner_train_rows = train_row_array[inner_train_pos]
early_stop_rows = train_row_array[early_stop_pos]
X_inner_train = X_train.loc[inner_train_rows]
y_inner_train = y_train.loc[inner_train_rows]
X_early_stop = X_train.loc[early_stop_rows]
y_early_stop = y_train.loc[early_stop_rows]

# 配置项之外，自动纳入由训练集预处理器识别出的名义类别列；档位保持有序数值。
_categorical_candidates = set(SAMPLING_CATEGORICAL_CANDIDATES)
SAMPLING_CATEGORICAL_FEATURES = [
    c for c in feature_names
    if c in _categorical_candidates or c in model_preprocessor.categorical_features_
]
print("预处理器高缺失删列:", model_preprocessor.dropped_high_missing_)
print("采样/LightGBM 按类别处理的字段:", SAMPLING_CATEGORICAL_FEATURES)

# 保留旧变量名，避免后续原有分析单元失效。
X_train_half, X_test_half = X_train, X_test
y_train_half, y_test_half = y_train, y_test

split_rows = []
for split_name, y_part in [
    ("训练集", y_train),
    ("模型选择验证集", y_val),
    ("概率校准集", y_cal),
    ("最终测试集", y_test),
]:
    split_rows.append({
        "集合": split_name,
        "客户数": len(y_part),
        "正样本数": int(y_part.sum()),
        "正样本率": float(y_part.mean()),
    })
split_summary_df = pd.DataFrame(split_rows)
display(split_summary_df)

inner_split_summary_df = pd.DataFrame([
    {"集合": "训练集内部拟合子集", "客户数": len(y_inner_train),
     "正样本数": int(y_inner_train.sum()), "正样本率": float(y_inner_train.mean())},
    {"集合": "训练集内部早停集", "客户数": len(y_early_stop),
     "正样本数": int(y_early_stop.sum()), "正样本率": float(y_early_stop.mean())},
])
display(inner_split_summary_df)

for split_name, y_part in [("训练集", y_train), ("模型选择验证集", y_val), ("概率校准集", y_cal), ("最终测试集", y_test)]:
    if y_part.nunique() < 2:
        raise ValueError(f"{split_name} 只含一个类别，无法训练或计算 AUC。")

for split_name, y_part in [("训练集内部拟合子集", y_inner_train), ("训练集内部早停集", y_early_stop)]:
    if y_part.nunique() < 2:
        raise ValueError(f"{split_name} 只含一个类别，无法执行内部早停。")

print(f"整体样本: {len(y):,}  正样本率={OVERALL_POS_RATE:.4%}  ({int(y.sum())} 正)")
print("四个集合按双标签联合分层随机划分；两个 TARGET 共用客户划分；采样只作用于模型训练数据。")
print("LightGBM早停只使用60%训练集的内部早停子集；模型选择验证集不参与早停。")
print("概率校准集不参与任务2模型训练或方法选择；最终测试按测试集正样本率对应分位数确定Top-ρ阈值。")


def _safe_precision_recall(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    rec = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
    pre = float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0
    return pre, rec


def _predict_model(model, X_eval):
    X_arr = X_eval.values if hasattr(X_eval, "values") else np.asarray(X_eval)
    if isinstance(model, (list, tuple)):
        probabilities = [m.predict(X_arr) for m in model]
        return np.mean(np.vstack(probabilities), axis=0)
    return model.predict(X_arr)


def evaluate_model(
    model, X_eval, y_eval, label, split_name="评估集",
    fixed_threshold=None, target_positive_rate=None, threshold_source=None,
):
    """计算排序与分类指标；最终测试阈值按测试集正样本率对应的预测概率分位数确定。"""
    prob = _predict_model(model, X_eval)
    y_arr = y_eval.values if hasattr(y_eval, "values") else np.asarray(y_eval)
    actual_rate = float(np.mean(y_arr))
    if fixed_threshold is None:
        target_positive_rate = (
            actual_rate if target_positive_rate is None else float(target_positive_rate)
        )
        threshold = float(np.quantile(prob, 1 - target_positive_rate))
        threshold_source = threshold_source or split_name
    else:
        threshold = float(fixed_threshold)
        target_positive_rate = (
            float(target_positive_rate) if target_positive_rate is not None else np.nan
        )
        threshold_source = threshold_source or "外部固定阈值"
    pred = (prob >= threshold).astype(int)
    predicted_positive_rate = float(np.mean(pred))

    precision, recall = _safe_precision_recall(y_arr, pred)
    # 兼容旧版scikit-learn：不向f1_score传入zero_division，直接用已安全计算的P/R求F1
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    beta = 2
    f2 = ((1 + beta**2) * precision * recall / (beta**2 * precision + recall)
          if (beta**2 * precision + recall) > 0 else 0.0)
    mcc = matthews_corrcoef(y_arr, pred)
    cm = confusion_matrix(y_arr, pred, labels=[0, 1])
    acc = accuracy_score(y_arr, pred)
    auc = roc_auc_score(y_arr, prob)
    auprc = average_precision_score(y_arr, prob)
    print(f"\n{'='*70}")
    print(f"  {split_name}评估：{label}")
    print(f"{'='*70}")
    print(f"  实际正样本率: {actual_rate:.4%}")
    print(f"  AUC={auc:.4f}  AUPRC={auprc:.4f}")
    print(f"  阈值={threshold:.6f}（来源: {threshold_source}）")
    print(f"  预测正样本率: {predicted_positive_rate:.4%}")
    print(f"  Precision={precision:.4f}  Recall={recall:.4f}  F1={f1:.4f}")
    print(f"  F2={f2:.4f}  MCC={mcc:.4f}  Accuracy={acc:.4f}")
    print(f"  混淆矩阵: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")

    return dict(
        label=label, split=split_name, auc=auc, auprc=auprc, baseline=actual_rate,
        recall=recall, precision=precision, f1=f1, f2=f2, mcc=mcc, acc=acc,
        threshold=threshold, threshold_positive_rate=target_positive_rate,
        threshold_source=threshold_source, predicted_positive_rate=predicted_positive_rate,
        tn=int(cm[0,0]), fp=int(cm[0,1]), fn=int(cm[1,0]), tp=int(cm[1,1]),
        prob=prob, pred=pred, y_te=y_arr,
    )


def plot_cumulative_gain(prob, y_true, title="累积增益图", save_path="cumulative_gain.png",
                         table_save_path="cumulative_gain_table.csv"):
    """按指定 Top K% 输出累积增益表，并以这些业务节点绘制平滑累积增益曲线。"""
    y_arr = y_true.values if hasattr(y_true, "values") else np.asarray(y_true)
    prob_arr = np.asarray(prob, dtype=float)
    if len(y_arr) != len(prob_arr):
        raise ValueError("累积增益图的预测概率与真实标签长度不一致。")
    if len(y_arr) == 0:
        raise ValueError("测试集为空，无法绘制累积增益图。")

    sort_idx = np.argsort(prob_arr)[::-1]
    y_sorted = y_arr[sort_idx]
    n = len(y_sorted)
    total_pos = int(y_sorted.sum())
    top_k_list = [1, 3, 5, 10, 20, 30, 50]

    rows = []
    for k in top_k_list:
        top_n = min(n, max(1, int(np.ceil(n * k / 100))))
        top_pos = int(y_sorted[:top_n].sum())
        recall_pct = top_pos / total_pos * 100 if total_pos > 0 else 0.0
        top_positive_rate = top_pos / top_n * 100
        rows.append({
            "Top K%": f"Top {k}%",
            "召回正样本（%）": recall_pct,
            "Top K%客户中真实正样本比例（%）": top_positive_rate,
        })

    gain_table = pd.DataFrame(rows)
    from IPython.display import display
    print("\n累积增益表（测试集）：")
    display(gain_table.style.format({
        "召回正样本（%）": "{:.2f}",
        "Top K%客户中真实正样本比例（%）": "{:.2f}",
    }))
    gain_table.to_csv(table_save_path, index=False, encoding="utf-8-sig")
    print(f"累积增益表已保存: {table_save_path}")

    # 以业务指定的 Top K% 节点绘制保形平滑曲线，避免逐客户连接造成折线感。
    x_nodes = np.array([0] + top_k_list + [100], dtype=float)
    y_nodes = np.array([0] + gain_table["召回正样本（%）"].tolist() + [100.0 if total_pos > 0 else 0.0])
    x_smooth = np.linspace(0, 100, 500)
    try:
        from scipy.interpolate import PchipInterpolator
        y_smooth = PchipInterpolator(x_nodes, y_nodes)(x_smooth)
    except ImportError:
        # 兼容未安装 scipy 的环境；仍只基于指定节点绘图。
        y_smooth = np.interp(x_smooth, x_nodes, y_nodes)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(x_smooth, y_smooth, color="steelblue", lw=2.5, label="模型累积增益曲线（指定Top K%节点平滑）")
    ax.scatter(x_nodes[1:-1], y_nodes[1:-1], color="steelblue", s=45, zorder=4, label="Top K%业务节点")
    ax.plot([0, 100], [0, 100], color="gray", lw=1.5, linestyle="--", label="随机基线（无模型）")
    for x, y_value in zip(x_nodes[1:-1], y_nodes[1:-1]):
        ax.annotate(f"Top {int(x)}%\n召回 {y_value:.1f}%", (x, y_value),
                    xytext=(4, -14), textcoords="offset points", fontsize=8, color="steelblue")
    ax.set_xlabel("覆盖客户比例（按预测概率从高到低，%）")
    ax.set_ylabel("召回正样本比例（%）")
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=9)
    ax.set_xlim([0, 100]); ax.set_ylim([0, 105]); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.show()
    print(f"图片已保存: {save_path}")
    return gain_table

print("\n✅ 双标签联合分层随机切分完成；evaluate_model / plot_cumulative_gain 已定义")


## Cell 13A：采样与 LightGBM 训练函数定义（快速路径必须运行）


In [ ]:
# ── Cell 13A：仅定义函数，不执行候选方法训练 ──
def _normalise_method_name(method):
    if method is None:
        return "baseline", "无采样基线"
    method_key = str(method).lower().replace("-", "_").replace(" ", "_")
    if method_key in ("ensemble", "easyensemble"):
        method_key = "easy_ensemble"
    elif method_key == "balancecascade":
        method_key = "balance_cascade"
    labels = {
        "balance_cascade": "Balance Cascade（分类器驱动）",
        "easy_ensemble": "Easy Ensemble（随机欠采样集成）",
        "smoteenn": "类别安全 SMOTE + ENN",
        "smotetomek": "类别安全 SMOTE + Tomek",
    }
    return method_key, labels.get(method_key, method_key)


def _sample_training_data(method_key, X_base, y_base):
    if method_key == "baseline":
        return (X_base, y_base)
    if method_key in ("balance_cascade", "easy_ensemble"):
        sampler = sampler_factory(
            method=method_key,
            random_state=RANDOM_STATE,
            n_estimators=SAMPLING_N_ESTIMATORS,
            ratio=SAMPLING_ENSEMBLE_RATIO,
            categorical_features=SAMPLING_CATEGORICAL_FEATURES,
        )
    else:
        sampler = sampler_factory(
            method=method_key,
            sampling_strategy=SAMPLING_STRATEGY,
            random_state=RANDOM_STATE,
            categorical_features=SAMPLING_CATEGORICAL_FEATURES,
        )
    if method_key == "balance_cascade":
        # 子集必须由随后真正训练并集成的 LightGBM 逐轮驱动，不能提前用代理模型生成。
        return sampler.initialize(X_base, y_base)
    return sampler.fit_resample(X_base, y_base)

def _train_one_lgb(Xtr_use, ytr_use, X_valid, y_valid, label, params_base, num_rounds=None):
    params_local = params_base.copy()
    params_local.pop("scale_pos_weight", None)
    params_local.pop("early_stopping_rounds", None)
    configured_rounds = params_local.pop("n_estimators", LGB_PARAMS["n_estimators"])
    rounds = int(configured_rounds if num_rounds is None else num_rounds)
    training_columns = list(Xtr_use.columns) if hasattr(Xtr_use, "columns") else feature_names
    categorical_features_lgb = [
        c for c in SAMPLING_CATEGORICAL_FEATURES if c in training_columns
    ]
    dtrain = lgb.Dataset(
        Xtr_use, label=ytr_use, feature_name=feature_names,
        categorical_feature=categorical_features_lgb,
    )

    if X_valid is not None:
        dvalid = lgb.Dataset(X_valid, label=y_valid, feature_name=feature_names, reference=dtrain)
        booster_one = lgb.train(
            params=params_local,
            train_set=dtrain,
            num_boost_round=rounds,
            valid_sets=[dtrain, dvalid],
            valid_names=["train", "validation"],
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose_eval=100,
        )
    else:
        booster_one = lgb.train(
            params=params_local,
            train_set=dtrain,
            num_boost_round=rounds,
            verbose_eval=False,
        )
    print(f"  {label} 迭代轮次: {booster_one.best_iteration or rounds}")
    return booster_one


def _train_sampled(sampled, X_valid, y_valid, scene_label, params_base, round_limits=None):
    def _rounds_for(model_index):
        if isinstance(round_limits, (list, tuple, np.ndarray)):
            if len(round_limits) == 0:
                return None
            # 开发集的有效级联轮数可能与训练集不同；超出时复用最后一个早停轮数。
            return round_limits[min(model_index, len(round_limits) - 1)]
        return round_limits

    if isinstance(sampled, BalanceCascade):
        models = []
        while sampled.has_next_subset():
            model_index = len(models)
            X_sub, y_sub = sampled.next_subset()
            rounds = _rounds_for(model_index)
            model = _train_one_lgb(
                X_sub, y_sub, X_valid, y_valid,
                f"{scene_label} | model {model_index+1}/{sampled.effective_n_estimators_}",
                params_base, rounds,
            )
            models.append(model)
            # 用本轮实际 LightGBM 在当前多数类池上的分数调整阈值并删除易分样本。
            remaining_prob = model.predict(sampled.remaining_X())
            cascade_info = sampled.update(remaining_prob, score_label=1)
            if cascade_info["pruned_for_next_stage"]:
                print(
                    f"  Cascade stage {cascade_info['stage']}: "
                    f"目标FPR={cascade_info['target_fpr']:.4f}, "
                    f"阈值={cascade_info['threshold']:.6f}, "
                    f"下一轮多数类={cascade_info['remaining_majority']:,}"
                )
            else:
                print(f"  Cascade stage {cascade_info['stage']}: 最后一轮完成，不再淘汰样本")
        return models
    if isinstance(sampled, list):
        models = []
        for model_index, (X_sub, y_sub) in enumerate(sampled):
            rounds = _rounds_for(model_index)
            models.append(_train_one_lgb(
                X_sub, y_sub, X_valid, y_valid,
                f"{scene_label} | model {model_index+1}/{len(sampled)}",
                params_base, rounds,
            ))
        return models
    X_sampled, y_sampled = sampled
    return _train_one_lgb(
        X_sampled, y_sampled, X_valid, y_valid, scene_label, params_base, round_limits
    )


def _best_rounds(model):
    def _one(model_one):
        return int(model_one.best_iteration or model_one.current_iteration() or LGB_PARAMS["n_estimators"])
    if isinstance(model, (list, tuple)):
        return [_one(m) for m in model]
    return _one(model)


## Cell 13B：三模式配置阶段（完整比较 / 只更新轮数 / 历史复用）


In [ ]:
# ── Cell 13B：可选，只在需要重新比较采样方法时运行 ──
if RUN_MODEL_SELECTION:
    all_results = []
    trained_boosters = {}
    methods_to_run = [None] + list(dict.fromkeys(SAMPLING_METHODS))
    # ── 候选方法：只在 D_tr 采样/训练，在原始 D_val 评价 ──
    for method_order, method in enumerate(methods_to_run):
        method_key, method_label = _normalise_method_name(method)
        print(f"\n{'━'*72}\n▶ 候选场景：{method_label}\n{'━'*72}")
        candidate_params = LGB_PARAMS.copy()
        candidate_params.pop("scale_pos_weight", None)

        # 阶段1：只在60%训练集内部确定早停轮数，避免模型选择验证集参与早停。
        sampled_inner = _sample_training_data(method_key, X_inner_train, y_inner_train)
        early_stop_model = _train_sampled(
            sampled_inner, X_early_stop, y_early_stop,
            f"{method_label} | 训练集内部早停", candidate_params,
        )
        candidate_rounds = _best_rounds(early_stop_model)

        # 阶段2：按固定轮数在完整60%训练集上重新采样、重新训练候选模型。
        # 原始15%模型选择验证集只在随后评价一次，不进入训练或早停。
        sampled_train = _sample_training_data(method_key, X_train, y_train)
        candidate_model = _train_sampled(
            sampled_train, None, None,
            f"{method_label} | 完整训练集", candidate_params, candidate_rounds,
        )

        sampled_subsets = (
            sampled_train.subsets_ if isinstance(sampled_train, BalanceCascade)
            else sampled_train
        )
        if isinstance(sampled_subsets, list):
            print(f"  返回 {len(sampled_subsets)} 个训练子集")
            for subset_index, (_, y_sub) in enumerate(sampled_subsets, start=1):
                print_sampling_summary(y_train, y_sub, f"{method_label} subset{subset_index}")
            sampled_positive_rate = float(
                np.mean([np.mean(y_sub) for _, y_sub in sampled_subsets])
            )
        else:
            _, y_sampled = sampled_subsets
            if method_key != "baseline":
                print_sampling_summary(y_train, y_sampled, method_label)
            sampled_positive_rate = float(np.mean(y_sampled))
        result = evaluate_model(
            candidate_model, X_val, y_val, method_label,
            split_name="模型选择验证集",
        )
        result.update({
            "method": method_label,
            "method_key": method_key,
            "method_order": method_order,
            "train_pos_rate_after_sampling": sampled_positive_rate,
        })
        all_results.append(result)
        trained_boosters[method_label] = candidate_model

    method_selection_df = pd.DataFrame([{
        "method": result["method"],
        "method_key": result["method_key"],
        "validation_auc": result["auc"],
        "validation_precision_at_pi": result["precision"],
        "validation_recall_at_pi": result["recall"],
        "validation_f1_at_pi": result["f1"],
        "validation_threshold_at_pi": result["threshold"],
        "train_pos_rate_after_sampling": result["train_pos_rate_after_sampling"],
        "method_order": result["method_order"],
    } for result in all_results])

    top3_indices = method_selection_df.sort_values(
        ["validation_auc", "method_order"],
        ascending=[False, True], kind="mergesort",
    ).head(min(3, len(method_selection_df))).index      # 先选AUC Top 3
    method_selection_df["auc_top3"] = method_selection_df.index.isin(top3_indices)
    selected_index = method_selection_df.loc[top3_indices].sort_values(
        ["validation_precision_at_pi", "validation_auc", "method_order"],
        ascending=[False, False, True], kind="mergesort",
    ).index[0]                                          # 再从中选val上precision最高的
    method_selection_df["selected"] = method_selection_df.index == selected_index
    selected_method_key = method_selection_df.loc[selected_index, "method_key"]
    selected_method_label = method_selection_df.loc[selected_index, "method"]

    for result in all_results:
        row = method_selection_df[method_selection_df["method"] == result["method"]].iloc[0]
        result["auc_top3"] = bool(row["auc_top3"])
        result["selected"] = bool(row["selected"])

    print("\nAUC Top 3：", method_selection_df.loc[method_selection_df["auc_top3"], "method"].tolist())
    print("最终选择：", selected_method_label)
    display(method_selection_df.drop(columns=["method_order"]))
    method_selection_df.to_csv(f"sampling_method_selection_{TARGET}.csv", index=False, encoding="utf-8-sig")

    # ── 开发集 = 训练集 ∪ 模型选择验证集（约占全部样本75%），重训最优方法 ──
    X_fit = pd.concat([X_train, X_val], axis=0)
    y_fit = pd.concat([y_train, y_val], axis=0)
    development_ratio = len(X_fit) / len(df_clean)
    print(f'开发集样本数: {len(X_fit):,}，占全部样本: {development_ratio:.2%}')
    selected_candidate = trained_boosters[selected_method_label]
    selected_rounds = _best_rounds(selected_candidate)
    selected_model_config = {
        "target": TARGET,
        "selected_method_key": selected_method_key,
        "selected_method_label": selected_method_label,
        "selected_rounds": selected_rounds,
        "final_params": {k: v for k, v in LGB_PARAMS.items() if k != "scale_pos_weight"},
        "config_source": "full_method_selection",
    }
    with open(SELECTED_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(selected_model_config, f, ensure_ascii=False, indent=2)
    print(f"✅ 最优方法配置已保存：{SELECTED_CONFIG_PATH}")
    print("   selected_method_key =", repr(selected_method_key))
    print("   selected_rounds =", repr(selected_rounds))

elif REFIT_BEST_ROUNDS_ONLY:
    if os.path.isfile(SELECTED_CONFIG_PATH):
        with open(SELECTED_CONFIG_PATH, "r", encoding="utf-8") as f:
            _previous_config = json.load(f)
        if _previous_config.get("target") != TARGET:
            raise ValueError("历史最佳配置的 TARGET 与当前 TARGET 不一致。")
        selected_method_key = _previous_config["selected_method_key"]
        selected_method_label = _previous_config["selected_method_label"]
        refit_params = dict(_previous_config.get("final_params", LGB_PARAMS))
        print(f"从历史配置读取已选方法：{selected_method_label}")
    else:
        selected_method_key = FORCED_METHOD_KEY
        selected_method_label = FORCED_METHOD_LABEL
        refit_params = LGB_PARAMS.copy()
        print(f"尚无历史配置，使用方法兜底：{selected_method_label}")
    refit_params.pop("scale_pos_weight", None)

    # 只使用60%模型训练集：其内部拟合子集采样/训练，内部早停集确定最佳轮数。
    sampled_inner = _sample_training_data(
        selected_method_key, X_inner_train, y_inner_train
    )
    rounds_model = _train_sampled(
        sampled_inner, X_early_stop, y_early_stop,
        f"仅更新最佳轮数 | {selected_method_label}", refit_params,
    )
    selected_rounds = _best_rounds(rounds_model)
    selected_model_config = {
        "target": TARGET,
        "selected_method_key": selected_method_key,
        "selected_method_label": selected_method_label,
        "selected_rounds": selected_rounds,
        "final_params": refit_params,
        "config_source": "refit_best_rounds_only",
    }
    with open(SELECTED_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(selected_model_config, f, ensure_ascii=False, indent=2)
    print(f"✅ 最佳轮数已更新并保存：{SELECTED_CONFIG_PATH}")
    print("   selected_rounds =", repr(selected_rounds))

else:
    if os.path.isfile(SELECTED_CONFIG_PATH):
        with open(SELECTED_CONFIG_PATH, "r", encoding="utf-8") as f:
            selected_model_config = json.load(f)
        if selected_model_config.get("target") != TARGET:
            raise ValueError("历史最佳配置的 TARGET 与当前 TARGET 不一致。")
        selected_method_key = selected_model_config["selected_method_key"]
        selected_method_label = selected_model_config["selected_method_label"]
        selected_rounds = selected_model_config["selected_rounds"]
        print(f"✅ 已完全复用历史配置：{SELECTED_CONFIG_PATH}")
    elif _load_saved_artifacts and os.path.isfile(METADATA_PATH):
        # 兼容旧版只保存了全量模型元数据、尚未单独保存 selected_model_config 的制品。
        with open(METADATA_PATH, "r", encoding="utf-8") as f:
            _legacy_metadata = json.load(f)
        selected_model_config = {
            "target": TARGET,
            "selected_method_key": _legacy_metadata["selected_method_key"],
            "selected_method_label": _legacy_metadata["selected_method_label"],
            "selected_rounds": _legacy_metadata["selected_rounds"],
            "final_params": _legacy_metadata.get("final_params", LGB_PARAMS),
            "config_source": "migrated_from_full_model_metadata",
        }
        selected_method_key = selected_model_config["selected_method_key"]
        selected_method_label = selected_model_config["selected_method_label"]
        selected_rounds = selected_model_config["selected_rounds"]
        with open(SELECTED_CONFIG_PATH, "w", encoding="utf-8") as f:
            json.dump(selected_model_config, f, ensure_ascii=False, indent=2)
        print(f"✅ 已从旧版全量模型元数据迁移配置：{SELECTED_CONFIG_PATH}")
    else:
        if FORCED_ROUNDS is None:
            raise FileNotFoundError(
                f"未找到 {SELECTED_CONFIG_PATH}，且 FORCED_ROUNDS=None。"
                "请先开启完整方法比较或只更新最佳轮数。"
            )
        selected_method_key = FORCED_METHOD_KEY
        selected_method_label = FORCED_METHOD_LABEL
        selected_rounds = FORCED_ROUNDS
        selected_model_config = {
            "target": TARGET,
            "selected_method_key": selected_method_key,
            "selected_method_label": selected_method_label,
            "selected_rounds": selected_rounds,
            "final_params": {k: v for k, v in LGB_PARAMS.items() if k != "scale_pos_weight"},
            "config_source": "forced_fallback",
        }
        print("⚠️ 未找到历史配置，临时使用 FORCED_* 兜底。")
    print(f"   方法：{selected_method_label}")
    print(f"   最佳轮数：{selected_rounds}")


## Cell 13C：按选定配置获得全量 SHAP 模型（auto / train / load）


In [ ]:
# ── Cell 13C：加载制品，或按固定最优方法在全量数据上训练并保存 ──
final_params = dict(selected_model_config.get("final_params", LGB_PARAMS))
final_params.pop("scale_pos_weight", None)

if _load_saved_artifacts:
    with open(METADATA_PATH, "r", encoding="utf-8") as f:
        model_metadata = json.load(f)
    if model_metadata.get("feature_names") != list(feature_names):
        raise ValueError("保存模型的 feature_names 与当前预处理结果不一致，请检查数据/配置或改用 train。")
    selected_method_key = model_metadata["selected_method_key"]
    selected_method_label = model_metadata["selected_method_label"]
    selected_rounds = model_metadata["selected_rounds"]
    final_params = dict(model_metadata.get("final_params", final_params))
    _loaded_models = []
    for i in range(int(model_metadata["model_count"])):
        model_path = os.path.join(MODEL_DIR, f"model_{i + 1}.txt")
        if not os.path.isfile(model_path):
            raise FileNotFoundError(f"模型成员缺失：{model_path}")
        _loaded_models.append(lgb.Booster(model_file=model_path))
    full_shap_model = _loaded_models[0] if len(_loaded_models) == 1 else _loaded_models
    print(f"✅ 已加载全量模型：{selected_method_label}，成员数={len(_loaded_models)}")
else:
    print(f"固定使用采样方法：{selected_method_label}")
    sampled_full = _sample_training_data(selected_method_key, X, y)
    full_shap_model = _train_sampled(
        sampled_full, None, None,
        f"全量最终模型 | {selected_method_label}",
        final_params, selected_rounds,
    )
    print("✅ 全量最终模型训练完成")

    os.makedirs(MODEL_DIR, exist_ok=True)
    models = list(full_shap_model) if isinstance(full_shap_model, (list, tuple)) else [full_shap_model]
    for i, model in enumerate(models):
        model.save_model(os.path.join(MODEL_DIR, f"model_{i + 1}.txt"))
    joblib.dump(model_preprocessor, PREPROCESSOR_PATH)
    model_metadata = {
        "target": TARGET,
        "selected_method_key": selected_method_key,
        "selected_method_label": selected_method_label,
        "selected_rounds": selected_rounds,
        "final_params": final_params,
        "selected_config_path": SELECTED_CONFIG_PATH,
        "feature_names": list(feature_names),
        "model_count": len(models),
        "random_state": RANDOM_STATE,
    }
    with open(METADATA_PATH, "w", encoding="utf-8") as f:
        json.dump(model_metadata, f, ensure_ascii=False, indent=2)
    print(f"✅ 已保存 {len(models)} 个模型到 {MODEL_DIR}")
    print(f"✅ 已保存预处理器：{PREPROCESSOR_PATH}")
    print(f"✅ 已保存元数据：{METADATA_PATH}")

# 兼容后续 Gain 单元使用的变量名。
models = list(full_shap_model) if isinstance(full_shap_model, (list, tuple)) else [full_shap_model]
final_model = full_shap_model
booster = models[0]
trained_boosters = globals().get("trained_boosters", {})
trained_boosters["全量最终模型"] = full_shap_model
params_run = final_params


## Cell 15：Gain 特征重要性 + 指定字段专项查询

In [ ]:
# ── Cell 15：字段中文名映射 + 工具函数 ──

FIELD_CN_MAP = {'cst_id': '客编', 'mar_sttn_cd': '婚姻状态', 'cst_star_cd': '客户星级代码', 'education_cd': '学历', 'age': '年龄', 'occup_cd': '职业', 'gnd_cd': '性别', 'bank_cust_become_date': '成为我行客户时间', '当前aum': '当前AUM', '当前lum': '当前LUM', '科技人才对应得分': '科技人才对应得分', 'kum分': 'KUM分', 'lum分': 'LUM分', 'aum分': 'AUM分', '总分': '总分', '档位': '档位', 'credamt': '授信额度', 'lastyracmmpblandmonum': '上年累计手机银行登录月数', 'yr_acm_mpb_land_cnt': '年累计登录次数', 'yr_acm_mpb_land_dys': '年累计手机银行登录天数', 'yr_acm_mpb_land_monum': '年累计手机银行登录月数', 'yracm_mpb_fncltx_dnum': '年累计手机银行账务性交易笔数', 'yr_acm_mpb_fncltx_amt': '年累计手机银行账务性交易金额', 'acgmocr12mampblandcnt': '按月口径近12个月累计登录次数', 'acgmocrr12mampblmonum': '按月口径近12个月累计登录月数', 'acgmocrr6mampbftxdnum': '按月口径近6个月账务性交易笔数', 'acgmoclrr6mampbftxamt': '按月口径近6个月账务性交易金额', 'acgmocrr3mampblandcnt': '按月口径近3个月累计登录次数', 'acgmocrr3mampblanddys': '按月口径近3个月累计登录天数', 'acgmocrr3mampbftxdnum': '按月口径近3个月账务性交易笔数', 'acgmoclrr3mampbftxamt': '按月口径近3个月账务性交易金额', 'lmth_acm_mpb_land_cnt': '上月累计手机银行登录次数', 'mo_acm_mpb_land_dys': '月累计手机银行登录天数', 'moacm_mpb_fncltx_dnum': '月累计手机银行账务性交易笔数', 'mo_acm_mpb_fncltx_amt': '月累计手机银行账务性交易金额', 'rt12mofcccpcyscorscor': '近12月美食餐饮消费偏好评分', 'rt12mobtccpcyscorscor': '近12月出行商旅消费偏好评分', 'rt12moceccpcyscorscor': '近12月文化娱乐消费偏好评分', 'rt12mosepcccpcscorscor': '近12月安居置业消费偏好评分', 'rt12molsccpcyscorscor': '近12月生活购物消费偏好评分', 'rt12mocolccpcscorscor': '近12月车主生活消费偏好评分', 'rt12mols0ccpcyscorscor': '近12月生活服务消费偏好评分', 'rt12momhccpcyscorscor': '近12月医疗健康消费偏好评分', 'rt12moesccpcyscorscor': '近12月教育学习消费偏好评分', 'rt12mowcccpcyscorscor': '近12月西式餐饮消费偏好评分', 'rt12mchncccpcscorscor': '近12月中式餐饮消费偏好评分', 'rt12mofffccpcscorscor': '近12月水果生鲜消费偏好评分', 'rt12mopccpcpyscorscor': '近12月糕点消费偏好评分', 'rt12mosccpcpyscorscor': '近12月零食消费偏好评分', 'rt12modccpcpyscorscor': '近12月饮品消费偏好评分', 'rt12motaccpcyscorscor': '近12月烟酒消费偏好评分', 'rt12motgccpcyscorscor': '近12月外卖团购消费偏好评分', 'rt12mobmfccpcscorscor': '近12月公交地铁轮渡消费偏好评分', 'rt12mosbccpcyscorscor': '近12月共享单车消费偏好评分', 'rt12motccpcpyscorscor': '近12月打车消费偏好评分', 'rt12mocrccpcyscorscor': '近12月租车消费偏好评分', 'rt12mottccpcyscorscor': '近12月火车票消费偏好评分', 'rt12moatccpcyscorscor': '近12月机票消费偏好评分', 'rt12moghhccpcscorscor': '近12月宾馆酒店消费偏好评分', 'rt12motwccpcyscorscor': '近12月旅行网站消费偏好评分', 'rt12moctccpcyscorscor': '近12月邮轮游消费偏好评分', 'rt12modc0ccpcpyscorscor': '近12月下载消费偏好评分', 'rt12mosc0ccpcpyscorscor': '近12月社交消费偏好评分', 'rt12moicccpcyscorscor': '近12月网吧消费偏好评分', 'rt12movwccpcyscorscor': '近12月视频网站消费偏好评分', 'rt12molccpcpyscorscor': '近12月直播消费偏好评分', 'rt12momccpcpyscorscor': '近12月电影消费偏好评分', 'rt12moktvccpcscorscor': '近12月K歌消费偏好评分', 'rt12mogccpcpyscorscor': '近12月游戏消费偏好评分', 'rt12moaccpcpyscorscor': '近12月动漫消费偏好评分', 'rt12mufoocccpcscorscor': '近12月娃娃机消费偏好评分', 'rt12morccpcpyscorscor': '近12月游乐场消费偏好评分', 'rt12mosc1ccpcpyscorscor': '近12月演出消费偏好评分', 'rt12moaoacccpcscorscor': '近12月户外活动消费偏好评分', 'rt12moomccpcyscorscor': '近12月在线音乐消费偏好评分', 'rt12mohbmccpcscorscor': '近12月家居建材消费偏好评分', 'rt12mohprccpcscorscor': '近12月房产租赁消费偏好评分', 'rt12moshtccpcscorscor': '近12月二手交易消费偏好评分', 'rt12mosc2ccpcpyscorscor': '近12月商超消费偏好评分', 'rt12mocccpcpyscorscor': '近12月便利店消费偏好评分', 'rt12momdcccpcscorscor': '近12月美妆日化消费偏好评分', 'rt12moavmccpcscorscor': '近12月自动售货机消费偏好评分', 'rt12mosc3ccpcpyscorscor': '近12月文具消费偏好评分', 'rt12molc4ccpcpyscorscor': '近12月彩票消费偏好评分', 'rt12moedlccpcscorscor': '近12月快递物流消费偏好评分', 'rt12moaaccpcyscorscor': '近12月服装配饰消费偏好评分', 'rt12mccpgccpcscorscor': '近12月文创杂货消费偏好评分', 'rt12mmpncdccpccscrscor': '近12月手机数码消费偏好评分', 'rt12moac0ccpcpyscorscor': '近12月家电消费偏好评分', 'rt12mopc1ccpcpyscorscor': '近12月宠物消费偏好评分', 'rt12mostvcccpcscorscor': '近12月小型车辆消费偏好评分', 'rt12moagccpcyscorscor': '近12月农业园艺消费偏好评分', 'rt12miamcccpcscorscor': '近12月母婴综合消费偏好评分', 'rt12mocscccpcyscorscor': '近12月汽车销售消费偏好评分', 'rt12mshctccpcscorscor': '近12月二手车交易消费偏好评分', 'rt12mogc2ccpcpyscorscor': '近12月加油消费偏好评分', 'rt12moddccpcyscorscor': '近12月代驾消费偏好评分', 'rt12mocc0ccpcpyscorscor': '近12月充电消费偏好评分', 'rt12mopfccpcyscorscor': '近12月停车费消费偏好评分', 'rt12mocaccpcyscorscor': '近12月汽车用品消费偏好评分', 'rt12motfccpcyscorscor': '近12月交通罚款消费偏好评分', 'rt12mcwmmccpcscorscor': '近12月洗车维修消费偏好评分', 'rt12mpthfccpcscorscor': '近12月通行费消费偏好评分', 'rt12moaiccpcyscorscor': '近12月车险消费偏好评分', 'rt12mopnccpcyscorscor': '近12月车牌消费偏好评分', 'rt12momnccpcyscorscor': '近12月地图导航消费偏好评分', 'rt12modtccpcyscorscor': '近12月驾考消费偏好评分', 'rt12mowfccpcyscorscor': '近12月水费消费偏好评分', 'rt12moefccpcyscorscor': '近12月电费消费偏好评分', 'rt12mogfccpcyscorscor': '近12月燃气消费偏好评分', 'rt12mohfccpcyscorscor': '近12月供暖消费偏好评分', 'rt12mocfccpcyscorscor': '近12月有线电视消费偏好评分', 'rt12mobfccpcyscorscor': '近12月宽带消费偏好评分', 'rt12motcrccpcscorscor': '近12月话费充值消费偏好评分', 'rt12mopf0ccpcyscorscor': '近12月党费消费偏好评分', 'rt12mopf1ccpcyscorscor': '近12月物业费消费偏好评分', 'rt12moftfccpcscorscor': '近12月财政税费消费偏好评分', 'rt12mosccccpcyscorscor': '近12月共享充电消费偏好评分', 'rt12molc2ccpcpyscorscor': '近12月洗衣消费偏好评分', 'rt12mopc0ccpcpyscorscor': '近12月打印消费偏好评分', 'rt12mohsccpcyscorscor': '近12月美发造型消费偏好评分', 'rt12morjhccpcscorscor': '近12月招聘求职消费偏好评分', 'rt12moapccpcyscorscor': '近12月广告策划消费偏好评分', 'rt12mohccpcpyscorscor': '近12月家政服务消费偏好评分', 'rt12moinccpcyscorscor': '近12月月嫂消费偏好评分', 'rt12moeebccpcscorscor': '近12月出入境业务消费偏好评分', 'rt12mopwcccpcscorscor': '近12月公益众筹消费偏好评分', 'rt12mohcccpcyscorscor': '近12月医院诊所消费偏好评分', 'rt12mopc2ccpcpyscorscor': '近12月药店消费偏好评分', 'rt12mhchpccpcscorscor': '近12月保健养生消费偏好评分', 'rt12mofccpcpyscorscor': '近12月健身消费偏好评分', 'rt12mosmcccpcscorscor': '近12月共享按摩椅消费偏好评分', 'rt12mog0ccpcpyscorscor': '近12月敬老院消费偏好评分', 'rt12mo1ccpcpyscorscor': '近12月疗养院消费偏好评分', 'rt12momscccpcscorscor': '近12月月子中心消费偏好评分', 'rt12mopccccpcyscorscor': '近12月体检中心消费偏好评分', 'rt12mop0ccpcyscorscor': '近12月整形医美消费偏好评分', 'rt12mos1cpcpyscorscor': '近12月学校消费偏好评分', 'rt12motecccpcyscorscor': '近12月培训考试消费偏好评分', 'rt12moosaccpcscorscor': '近12月出国留学消费偏好评分', 'rt12mobccpcpyscorscor': '近12月图书消费偏好评分', 'rt12moolrccpcscorscor': '近12月线上阅读消费偏好评分', 'rt12moat0ccpcyscorscor': '近12月学术论文消费偏好评分', 'rt12mokqccpcyscorscor': '近12月知识问答消费偏好评分'}

# 合并本次Excel提供的86个消费行为字段映射（函数内部键已统一为小写）
FIELD_CN_MAP.update(get_consumption_field_cn_map())
# 补充模型衍生字段及手机银行字段的中文名称
FIELD_CN_MAP.update({
    'bank_cust_become_date': '成为我行客户时间',
    'days_since_become_cust': '成为我行客户天数',
    'lblclbrmpbactvcst_ind': '手机银行活跃客户标识',
    'credamt_sq': '授信额度平方项',
    'credamt_cube': '授信额度立方项',
    'credamt_log': '授信额度对数项',
    'prvtcstr12mvvtccmdnum': '对私客户近十二个月车船运输类消费笔数',
    'prvtcstr12mvvtccnpamt': '对私客户近十二个月车船运输类消费金额',
})

def _get_cn(fname):
    """返回字段中文名，找不到则返回原英文名"""
    return FIELD_CN_MAP.get(str(fname).lower(), fname)

def _show_fields(imp_df, fields, title=""):
    """按字段名查询SHAP排名，展示中英文对照"""
    if title:
        print(f"\n===== {title} =====")
    found, notfound = [], []
    for f in fields:
        row = imp_df[imp_df["feature"].str.lower() == str(f).lower()]
        if len(row) > 0:
            found.append({
                "中文名":      _get_cn(row.iloc[0]["feature"]),
                "英文名":      row.iloc[0]["feature"],
                "mean_abs_shap": row.iloc[0].get("mean_abs_shap", row.iloc[0].get("importance", 0)),
                "全特征排名":  row.iloc[0].get("shap_rank", row.iloc[0].get("rank", "-")),
            })
        else:
            notfound.append(f)
    if found:
        display(pd.DataFrame(found).sort_values("全特征排名").reset_index(drop=True))
    if notfound:
        print(f"  以下字段不在特征集中（可能未纳入建模）: {notfound}")

# 手机银行字段列表（供 Cell 16 专项查询使用）
MPB_FIELDS = [
    "lblclbrmpbactvcst_ind","yr_acm_mpb_land_cnt","lmth_acm_mpb_land_cnt",
    "acgmocrr3mampblandcnt","acgmocr12mampblandcnt",
    "mo_acm_mpb_land_dys","acgmocrr12mampblmonum","yr_acm_mpb_land_monum",
    "lastyracmmpblandmonum","moacm_mpb_fncltx_dnum","mo_acm_mpb_fncltx_amt",
    "acgmocrr3mampblanddys","yr_acm_mpb_land_dys","yracm_mpb_fncltx_dnum",
    "yr_acm_mpb_fncltx_amt","acgmocrr3mampbftxdnum","acgmoclrr3mampbftxamt",
    "acgmocrr6mampbftxdnum","acgmoclrr6mampbftxamt",
]

print(f"✅ FIELD_CN_MAP 已加载，共 {len(FIELD_CN_MAP)} 个字段映射")
print(f"   其中消费偏好字段: {sum(1 for k in FIELD_CN_MAP if '月' in FIELD_CN_MAP[k] and '偏好' in FIELD_CN_MAP[k])} 个")


## Cell 16：全量模型的概率贡献 SHAP + 年龄/性别可视化 + 累积增益图

使用数值稳定 sigmoid；成员模型分别计算概率贡献后再聚合，并校验贡献加和能还原预测概率。


In [ ]:
# ── Cell 16：SHAP 分析 + 性别分组违约率可视化 + 累积增益图 ──
#
# 设计原则：
#   - 直接解释 Cell 13C 在全量数据上训练或加载的 full_shap_model
#   - SHAP 客户解释池仍为原始全量客户，与模型训练客户池是两个不同概念
#   - 累积增益图：双标签联合分层随机 60/15/15/10 划分，最终模型在测试集绘制
#   - 特征名统一显示中文名称
#   - 标题根据 TARGET 自动切换（y_freq=频繁支用 / y_dq_risk=违约预测）
#   - 性别分组只展示平均预测概率，不展示真实平均概率

# ── 标题前缀（根据任务自动切换）──
TASK_TITLE = "频繁支用预测任务" if TARGET == "y_freq" else "违约概率预测任务"

# ── 字段中文名映射 ──
_FIELD_CN = FIELD_CN_MAP  # 从 Cell 15 继承

def _cn(fname):
    return _FIELD_CN.get(str(fname).lower(), fname)

# SHAP绘图前强制核验：所有英文特征必须存在中文映射，避免图中漏出英文名。
_unmapped_shap_features = [
    f for f in feature_names
    if all(ord(ch) < 128 for ch in str(f)) and str(f).lower() not in _FIELD_CN
]
if _unmapped_shap_features:
    raise KeyError(f'以下SHAP特征缺少中文映射，请先补充FIELD_CN_MAP: {_unmapped_shap_features}')

# ════════════════════════════════════════════════════
# Step 1：对接全量最终模型（与测试指标、累积增益和 Gain 完全同一模型）
# ═════════════════════════════════════════════════════════════════════
print(f"[Step 1] SHAP 对接全量最终模型（任务：{TASK_TITLE}）...")
shap_model = full_shap_model
print(f"  模型：full_shap_model | {selected_method_label}（不重新训练）")
print(f"  原始全量客户解释池: {len(X):,}  正样本率: {y.mean():.4%}")

# 原始全量客户逐行输入全量最终模型；不对 SHAP 客户池再次抽样。
X_shap = X.copy()
y_shap = y.copy()
X_arr = X_shap.values if hasattr(X_shap, "values") else np.asarray(X_shap)

def _stable_sigmoid(values):
    """数值稳定的 sigmoid，兼容内部 Python 3.6 环境。"""
    values = np.asarray(values, dtype=float)
    result = np.empty_like(values)
    positive = values >= 0
    result[positive] = 1.0 / (1.0 + np.exp(-values[positive]))
    exp_values = np.exp(values[~positive])
    result[~positive] = exp_values / (1.0 + exp_values)
    return result

def _one_model_probability_contributions(model_one, x_array):
    """
    使用 LightGBM 原生 TreeSHAP（raw score）并映射到概率贡献。

    LightGBM 的 pred_contrib=True 返回原始分数空间贡献；这里沿每个样本的
    sigmoid 割线等比例映射各特征贡献，使映射后的贡献与该模型实际预测概率严格加和。
    该实现不依赖 shap 包，并保留原始 TreeSHAP 的特征贡献相对比例；它是
    可严格还原预测概率的链接函数映射，不等同于重新枚举概率输出的完整 Shapley 组合。
    """
    raw_contrib = np.asarray(model_one.predict(x_array, pred_contrib=True), dtype=float)
    expected_columns = x_array.shape[1] + 1
    if raw_contrib.ndim != 2 or raw_contrib.shape != (x_array.shape[0], expected_columns):
        raise ValueError(
            "LightGBM pred_contrib 输出形状异常：期望 (%d, %d)，实际 %s"
            % (x_array.shape[0], expected_columns, raw_contrib.shape)
        )

    raw_score = np.asarray(
        model_one.predict(x_array, raw_score=True), dtype=float
    ).reshape(-1)
    raw_reconstructed = raw_contrib.sum(axis=1)
    raw_error = float(np.max(np.abs(raw_reconstructed - raw_score)))
    if (not np.isfinite(raw_error)) or raw_error > 1e-5:
        raise ValueError("原始 TreeSHAP 加和校验失败，最大误差=%.3e" % raw_error)

    probability = np.asarray(model_one.predict(x_array), dtype=float).reshape(-1)
    raw_base = raw_contrib[:, -1]
    base_probability = _stable_sigmoid(raw_base)
    raw_delta = raw_score - raw_base
    probability_delta = probability - base_probability

    scale = np.empty_like(raw_delta)
    regular = np.abs(raw_delta) > 1e-12
    scale[regular] = probability_delta[regular] / raw_delta[regular]
    scale[~regular] = base_probability[~regular] * (1.0 - base_probability[~regular])

    probability_contrib = raw_contrib.copy()
    probability_contrib[:, :-1] *= scale[:, None]
    probability_contrib[:, -1] = base_probability
    # 吸收极小浮点残差，确保每行贡献之和与模型实际概率完全同口径。
    probability_contrib[:, -1] += probability - probability_contrib.sum(axis=1)
    probability_error = float(np.max(np.abs(probability_contrib.sum(axis=1) - probability)))
    if (not np.isfinite(probability_error)) or probability_error > 1e-8:
        raise ValueError("概率贡献加和校验失败，最大误差=%.3e" % probability_error)
    return probability_contrib, raw_error, probability_error

def _shap_contributions(model, x_array):
    """按实际预测规则聚合集成成员，返回可加和到预测概率的贡献矩阵。"""
    models = list(model) if isinstance(model, (list, tuple)) else [model]
    if not models:
        raise ValueError("SHAP 模型列表不能为空")

    combined = None
    member_raw_errors = []
    member_probability_errors = []
    for model_one in models:
        member_contrib, raw_error, probability_error = (
            _one_model_probability_contributions(model_one, x_array)
        )
        combined = member_contrib if combined is None else combined + member_contrib
        member_raw_errors.append(raw_error)
        member_probability_errors.append(probability_error)

    combined /= float(len(models))
    actual_probability = np.asarray(_predict_model(model, x_array), dtype=float).reshape(-1)
    ensemble_error = float(np.max(np.abs(combined.sum(axis=1) - actual_probability)))
    if (not np.isfinite(ensemble_error)) or ensemble_error > 1e-8:
        raise ValueError("集成 SHAP 概率贡献加和校验失败，最大误差=%.3e" % ensemble_error)

    print(
        "  SHAP加和审计: 成员数=%d, raw最大误差=%.3e, 概率最大误差=%.3e, 集成最大误差=%.3e"
        % (len(models), max(member_raw_errors), max(member_probability_errors), ensemble_error)
    )
    return combined

# ═════════════════════════════════════════════════════════════════════
# Step 2：计算原始全量客户的 SHAP 概率贡献
# ═════════════════════════════════════════════════════════════════════
print(f"\n[Step 2] 将原始全量客户输入全量最终模型并计算 SHAP 概率贡献（{len(X_shap):,} 个）...")
contrib_full = _shap_contributions(shap_model, X_arr)
shap_values = contrib_full[:, :-1]
base_probability = float(np.mean(contrib_full[:, -1]))
print(f"  ✅ SHAP 概率贡献计算完成  shape={shap_values.shape}  base_probability={base_probability:.4f}")

# ═════════════════════════════════════════════════════════════════════
# Step 3：从预处理器获取类别型特征，在 Top 20 Bar 与 Beeswarm 中剔除
# ════════════════════════════════════════════════════
CAT_FEATURES = {
    str(c).lower()
    for c in set(model_preprocessor.categorical_features_) | set(SAMPLING_CATEGORICAL_FEATURES)
}
fn_lower   = [f.lower() for f in feature_names]
cont_idx   = [i for i, f in enumerate(feature_names) if f.lower() not in CAT_FEATURES]
cont_names = [feature_names[i] for i in cont_idx]
shap_cont  = shap_values[:, cont_idx]
X_cont     = X_arr[:, cont_idx]

# ════════════════════════════════════════════════════
# Step 4：SHAP 统计汇总（中文名）
# ════════════════════════════════════════════════════
mean_abs_shap = np.abs(shap_values).mean(axis=0)
mean_shap     = shap_values.mean(axis=0)

shap_df = pd.DataFrame({
    "feature":    feature_names,
    "feature_cn": [_cn(f) for f in feature_names],
    "mean_abs_shap": mean_abs_shap,
    "mean_shap":     mean_shap,
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
shap_df["shap_rank"] = shap_df.index + 1

shap_cont_df = shap_df[~shap_df["feature"].str.lower().isin(CAT_FEATURES)].copy()
shap_cont_df = shap_cont_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
shap_cont_df["continuous_rank"] = shap_cont_df.index + 1

print(f"\n===== SHAP Top {SHAP_TOP_N} 非类别特征（{TASK_TITLE}）=====")
print(shap_cont_df.head(SHAP_TOP_N)[["continuous_rank","feature_cn","feature","mean_abs_shap","mean_shap"]].to_string(index=False))
print("  mean_shap > 0 = 该特征平均推高正类预测概率；< 0 = 平均压低")

# ════════════════════════════════════════════════════
# 密度感知抖动函数
# ════════════════════════════════════════════════════
def _density_jitter(vals, max_width=0.4, n_bins=50):
    jitter = np.zeros(len(vals))
    if len(vals) == 0: return jitter
    v_min, v_max = vals.min(), vals.max()
    if v_min == v_max: return jitter
    bins   = np.linspace(v_min, v_max, n_bins + 1)
    assign = np.clip(np.digitize(vals, bins) - 1, 0, n_bins - 1)
    max_cnt = max(np.bincount(assign).max(), 1)
    rng_j   = np.random.RandomState(RANDOM_STATE)
    for b in np.unique(assign):
        mask  = assign == b
        cnt   = mask.sum()
        width = max_width * (cnt / max_cnt)
        jitter[mask] = rng_j.uniform(-width, width, size=cnt)
    return jitter

cmap = plt.cm.RdBu_r

# ════════════════════════════════════════════════════
# Step 5：图1 —— SHAP Bar（中文名，剔除类别型特征）
# ════════════════════════════════════════════════════
print(f"\n[图1] SHAP Bar | {TASK_TITLE}")
top_n      = shap_cont_df.head(SHAP_TOP_N)
colors_bar = ["#d45f5f" if v > 0 else "#5f8dd4" for v in top_n["mean_shap"][::-1]]
fig, ax    = plt.subplots(figsize=(11, 7))
bars = ax.barh(top_n["feature_cn"][::-1], top_n["mean_abs_shap"][::-1],
               color=colors_bar, edgecolor="white")
for bar, val, mv in zip(bars, top_n["mean_abs_shap"][::-1], top_n["mean_shap"][::-1]):
    direction = "↑正类概率" if mv > 0 else "↓正类概率"
    ax.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2,
            f"{val:.4f} ({direction})", va="center", fontsize=8)
ax.set_xlabel("平均 |SHAP概率贡献|")
ax.set_title(f"SHAP特征重要性（剔除类别型特征） | {TASK_TITLE} | Top {SHAP_TOP_N}\n红=平均推高正类概率，蓝=平均压低")
plt.tight_layout()
plt.savefig(f"shap_bar_non_categorical_{TARGET}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"图片已保存: shap_bar_non_categorical_{TARGET}.png")

# ════════════════════════════════════════════════════
# Step 6：图2 —— SHAP Beeswarm（连续型，中文名，密度感知）
# ════════════════════════════════════════════════════
print(f"\n[图2] SHAP Beeswarm | {TASK_TITLE}")
cont_abs       = np.abs(shap_cont).mean(axis=0)
cont_order     = np.argsort(cont_abs)[::-1][:SHAP_TOP_N]
top_cont_names = [cont_names[i] for i in cont_order]
top_cont_cn    = [_cn(cont_names[i]) for i in cont_order]
shap_bee       = shap_cont[:, cont_order]
X_bee          = X_cont[:, cont_order]

fig, ax = plt.subplots(figsize=(12, max(6, len(top_cont_names)*0.5)))
for rank in range(len(top_cont_names)):
    shap_col = shap_bee[:, rank]
    x_col    = X_bee[:, rank]
    valid    = ~np.isnan(x_col)
    p1, p99  = (np.percentile(x_col[valid], [1, 99]) if valid.sum() > 1 else (0, 1))
    x_norm   = np.clip((x_col - p1) / (p99 - p1 + 1e-9), 0, 1)
    y_jitter = _density_jitter(shap_col, max_width=0.38)
    ax.scatter(shap_col, rank + y_jitter,
               c=x_norm, cmap=cmap, s=8, alpha=0.6, linewidths=0, vmin=0, vmax=1)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.01)
cbar.set_label("特征值（蓝=低  红=高）", fontsize=9)
cbar.set_ticks([0, 1]); cbar.set_ticklabels(["低", "高"])
ax.axvline(x=0, color="black", lw=0.8, linestyle="--", alpha=0.5)
ax.set_yticks(range(len(top_cont_names)))
ax.set_yticklabels(top_cont_cn, fontsize=9)
ax.set_xlabel("SHAP概率贡献（正=推高正类概率，负=压低）")
ax.set_title(f"SHAP Beeswarm | {TASK_TITLE} | 连续型特征 Top {SHAP_TOP_N}\n（类别型特征已移除，性别见图4/5）")
plt.tight_layout()
plt.savefig(f"shap_beeswarm_{TARGET}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"图片已保存: shap_beeswarm_{TARGET}.png")

# ════════════════════════════════════════════════════
# Step 7：图3 —— 手机银行字段专项 Beeswarm（中文名）
# ════════════════════════════════════════════════════
print(f"\n[图3] 手机银行字段 SHAP 专项 | {TASK_TITLE}")
mpb_fields_lower = [
    "lmth_acm_mpb_land_cnt","acgmocrr3mampblandcnt","acgmocr12mampblandcnt",
    "mo_acm_mpb_land_dys","acgmocrr12mampblmonum","yr_acm_mpb_land_monum",
    "lastyracmmpblandmonum","moacm_mpb_fncltx_dnum","mo_acm_mpb_fncltx_amt",
    "acgmocrr3mampblanddys","yr_acm_mpb_land_dys","yracm_mpb_fncltx_dnum",
    "yr_acm_mpb_fncltx_amt","acgmocrr3mampbftxdnum","acgmoclrr3mampbftxamt",
    "acgmocrr6mampbftxdnum","acgmoclrr6mampbftxamt",
]
mpb_idx     = [i for i, f in enumerate(feature_names) if f.lower() in mpb_fields_lower]
mpb_names   = [feature_names[i] for i in mpb_idx]
missing_mpb = [f for f in mpb_fields_lower if f not in fn_lower]
if missing_mpb:
    print(f"  以下字段不在特征集中，已跳过: {missing_mpb}")

if mpb_idx:
    shap_mpb      = shap_values[:, mpb_idx]
    X_mpb         = X_arr[:, mpb_idx]
    n_mpb         = len(mpb_names)
    mpb_abs_order = np.argsort(np.abs(shap_mpb).mean(axis=0))[::-1]
    shap_mpb      = shap_mpb[:, mpb_abs_order]
    X_mpb         = X_mpb[:, mpb_abs_order]
    mpb_names_sorted    = [mpb_names[i] for i in mpb_abs_order]
    mpb_cn_sorted       = [_cn(mpb_names[i]) for i in mpb_abs_order]

    fig, ax = plt.subplots(figsize=(12, max(5, n_mpb*0.55)))
    for rank in range(n_mpb):
        shap_col = shap_mpb[:, rank]
        x_col    = X_mpb[:, rank]
        valid    = ~np.isnan(x_col)
        p1, p99  = (np.percentile(x_col[valid], [1, 99]) if valid.sum() > 1 else (0, 1))
        x_norm   = np.clip((x_col - p1) / (p99 - p1 + 1e-9), 0, 1)
        y_jitter = _density_jitter(shap_col, max_width=0.38)
        ax.scatter(shap_col, rank + y_jitter,
                   c=x_norm, cmap=cmap, s=8, alpha=0.6, linewidths=0, vmin=0, vmax=1)
    sm3 = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
    sm3.set_array([])
    cbar3 = plt.colorbar(sm3, ax=ax, pad=0.01)
    cbar3.set_label("特征值（蓝=低  红=高）", fontsize=9)
    cbar3.set_ticks([0, 1]); cbar3.set_ticklabels(["低", "高"])
    ax.axvline(x=0, color="black", lw=0.8, linestyle="--", alpha=0.5)
    ax.set_yticks(range(n_mpb))
    ax.set_yticklabels(mpb_cn_sorted, fontsize=9)
    ax.set_xlabel("SHAP概率贡献（正=推高正类概率，负=压低）")
    ax.set_title(f"SHAP 手机银行字段专项 | {TASK_TITLE}")
    plt.tight_layout()
    plt.savefig(f"shap_mpb_{TARGET}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"图片已保存: shap_mpb_{TARGET}.png")

    print(f"\n[图3B] 手机银行字段 SHAP 方向柱状图 | {TASK_TITLE}")
    mpb_shap_df = shap_df[shap_df["feature"].str.lower().isin(mpb_fields_lower)].copy()
    mpb_plot_df = mpb_shap_df.sort_values("mean_abs_shap", ascending=True)
    fig, ax = plt.subplots(figsize=(12, max(5, len(mpb_plot_df)*0.55)))
    direction_colors = ["#d45f5f" if v >= 0 else "#5f8dd4" for v in mpb_plot_df["mean_shap"]]
    bars = ax.barh(mpb_plot_df["feature_cn"], mpb_plot_df["mean_shap"], color=direction_colors)
    ax.axvline(0, color="black", lw=0.9)
    for bar, value in zip(bars, mpb_plot_df["mean_shap"]):
        ax.text(value, bar.get_y()+bar.get_height()/2, f" {value:.4f}",
                va="center", ha="left" if value >= 0 else "right", fontsize=8)
    ax.set_xlabel("平均 SHAP概率贡献（正=推高正类概率，负=压低）")
    ax.set_title(f"手机银行字段影响方向 | {TASK_TITLE}\n红=正向影响，蓝=负向影响")
    plt.tight_layout()
    plt.savefig(f"shap_mpb_direction_bar_{TARGET}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"图片已保存: shap_mpb_direction_bar_{TARGET}.png")

    print(f"\n===== 手机银行字段 SHAP 数值表（{TASK_TITLE}）=====")
    display(mpb_shap_df[["shap_rank","feature_cn","feature","mean_abs_shap","mean_shap"]].reset_index(drop=True))

# ════════════════════════════════════════════════════
# Step 8：年龄层正样本构成饼图
# 数据来源：原始全量客户。预测标签阈值为预测概率的 (1-原始全量正样本率) 分位数。
# ════════════════════════════════════════════════════
print(f"\n===== 年龄层正样本构成分析 | {TASK_TITLE} =====")

age_shap_df = shap_df[shap_df["feature"].str.lower() == "age"].copy()
if not age_shap_df.empty:
    age_rank = int(age_shap_df["shap_rank"].iloc[0])
    print(f"  年龄（age）在全特征 SHAP 重要性中的排名：第 {age_rank} / {len(feature_names)} 名")
    display(age_shap_df[["shap_rank", "feature_cn", "feature", "mean_abs_shap", "mean_shap"]].reset_index(drop=True))

_age_col = next((c for c in df_clean.columns if str(c).lower() == "age"), None)
if _age_col is None:
    print("  ⚠️ 清洗数据中不存在 age 字段，跳过年龄层饼图。")
else:
    age_all = pd.to_numeric(df_clean.loc[X.index, _age_col], errors="coerce")
    age_prob = _predict_model(shap_model, X)
    full_positive_rate = float(np.asarray(y).mean())
    age_prediction_threshold = float(np.quantile(age_prob, 1 - full_positive_rate))
    age_pred = (age_prob >= age_prediction_threshold).astype(int)
    age_group_labels = ["30岁以下", "30岁-40岁", "40岁-50岁", "50岁以上"]
    age_group = pd.cut(
        age_all, bins=[-np.inf, 30, 40, 50, np.inf],
        labels=age_group_labels, right=False,
    )
    age_analysis_df = pd.DataFrame({
        "年龄层": age_group.to_numpy(),
        "真实标签": np.asarray(y, dtype=int),
        "预测标签": age_pred,
    }).dropna(subset=["年龄层"])

    age_counts = (
        age_analysis_df.groupby("年龄层", observed=False)
        .agg(真实正样本数=("真实标签", "sum"), 预测正样本数=("预测标签", "sum"))
        .reindex(age_group_labels, fill_value=0)
        .reset_index()
    )
    age_counts["真实正样本构成占比"] = age_counts["真实正样本数"] / max(age_counts["真实正样本数"].sum(), 1)
    age_counts["预测正样本构成占比"] = age_counts["预测正样本数"] / max(age_counts["预测正样本数"].sum(), 1)
    print(f"  预测标签阈值={age_prediction_threshold:.6f}（全量正样本率 {full_positive_rate:.4%} 对应的 1-正样本率 分位数）")
    display(age_counts)

    colors_age = ["#6baed6", "#74c476", "#fd8d3c", "#9e9ac8"]
    actual_title = "真实频繁支用正样本" if TARGET == "y_freq" else "真实违约正样本"
    predicted_title = "预测频繁支用正样本" if TARGET == "y_freq" else "预测违约正样本"
    for count_col, title_text, file_tag in [
        ("真实正样本数", actual_title, "actual"),
        ("预测正样本数", predicted_title, "predicted"),
    ]:
        values = age_counts[count_col].to_numpy(dtype=float)
        if values.sum() <= 0:
            print(f"  ⚠️ {title_text}总数为0，跳过饼图。")
            continue
        fig, ax = plt.subplots(figsize=(8, 6.5))
        ax.pie(
            values, labels=age_counts["年龄层"], colors=colors_age,
            autopct=lambda pct: f"{pct:.1f}%\n({int(round(pct / 100 * values.sum())):,}人)",
            startangle=90, counterclock=False,
            wedgeprops={"edgecolor": "white", "linewidth": 1.2},
        )
        ax.set_title(f"四个年龄层{title_text}人数构成 | {TASK_TITLE}\n每块占全部正样本人数的比例")
        ax.axis("equal")
        plt.tight_layout()
        output_name = f"age_group_{file_tag}_positive_composition_{TARGET}.png"
        plt.savefig(output_name, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"图片已保存: {output_name}")

# ════════════════════════════════════════════════════
# Step 9：性别分组平均预测概率
# 仅展示预测平均概率，不计算或展示真实平均概率。
# ════════════════════════════════════════════════════
print(f"\n[性别分组平均预测概率] {TASK_TITLE}")

if "gnd_cd" in feature_names:
    gnd_encoded = X["gnd_cd"].to_numpy(dtype=int)
    gnd_encoder = model_preprocessor.label_encoders_.get("gnd_cd")
    code_to_raw_gender = (
        {i: str(value).strip() for i, value in enumerate(gnd_encoder.classes_)}
        if gnd_encoder is not None else {}
    )
    raw_to_gender = {"1": "男", "1.0": "男", "男": "男",
                     "2": "女", "2.0": "女", "女": "女"}
    gender_labels = np.array([
        raw_to_gender.get(code_to_raw_gender.get(code, ""), f"其他({code_to_raw_gender.get(code, code)})")
        for code in gnd_encoded
    ])
    gender_prob = _predict_model(shap_model, X)
    gender_df = pd.DataFrame({"性别": gender_labels, "预测概率": gender_prob})
    gender_stats = (
        gender_df.groupby("性别")
        .agg(客户数=("预测概率", "size"), 平均预测概率=("预测概率", "mean"))
        .reset_index()
    )
    print(f"\n  性别分类平均预测{'支用率' if TARGET == 'y_freq' else '违约率'}（仅预测概率）：")
    display(gender_stats.style.format({"平均预测概率": "{:.4%}"}))

    fig, ax = plt.subplots(figsize=(7, 5))
    colors_g = ["steelblue", "tomato", "green", "#9e9ac8"][:len(gender_stats)]
    bars = ax.bar(gender_stats["性别"], gender_stats["平均预测概率"] * 100,
                  color=colors_g, width=0.55, edgecolor="white")
    for bar, value in zip(bars, gender_stats["平均预测概率"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{value:.2%}", ha="center", va="bottom", fontweight="bold")
    ax.set_ylabel("平均预测概率 (%)")
    ax.set_title(f"各性别平均预测{'支用率' if TARGET == 'y_freq' else '违约率'} | {TASK_TITLE}\n仅展示预测平均概率")
    ax.set_ylim(bottom=0, top=max(gender_stats["平均预测概率"].max() * 100 * 1.18, 0.1))
    plt.tight_layout()
    plt.savefig(f"gender_mean_pred_probability_{TARGET}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"图片已保存: gender_mean_pred_probability_{TARGET}.png")
else:
    print("  ⚠️ gnd_cd 不在 feature_names 中，跳过性别分析")

# ════════════════════════════════════════════════════
# Step 10：累积增益图（y_freq 营销场景，测试集）
# ════════════════════════════════════════════════════
print(f"\n[图6] 累积增益图 | {TASK_TITLE}")

if TARGET in ("y_freq", "y_dq_risk"):
    print("  模型：开发集（训练集+模型选择验证集，约75%）重训后的全量最终模型")
    print("  绘图：独立10%测试集（X_test, y_test），按预测概率从高到低排序")

    if TARGET == "y_freq":
        print("  业务含义：圈选 Top K% 客户，能覆盖多少比例的真实频繁支用客户")
    else:
        print("  业务含义：识别风险最高的 Top K% 客户，能覆盖多少比例的真实违约客户")

    prob_test_gain = _predict_model(full_shap_model, X_test)

    cumulative_gain_table = plot_cumulative_gain(
        prob=prob_test_gain,
        y_true=y_test,
        title=f"累积增益图 | {TASK_TITLE} | 最终开发模型在独立10%测试集",
        save_path=f"cumulative_gain_{TARGET}_test.png",
        table_save_path=f"cumulative_gain_{TARGET}_test_table.csv",
    )
else:
    print(f"  当前任务 {TARGET} 不支持绘制累积增益图。")

# ═════════════════════════════════════════════════════════════════════
# Step 11：手机银行字段在全特征中的SHAP排名专项展示
# ════════════════════════════════════════════════════
print(f"\n===== 手机银行特征在全部特征中的SHAP重要性排名 | {TASK_TITLE} =====")
print("  （排名基于全量最终模型对原始全量客户计算的 mean_abs_shap，排名数值越小=越重要）")
_show_fields(shap_df, MPB_FIELDS, title="")
print()
# 同时打印总特征数作为参考
print(f"  当前建模特征总数: {len(feature_names)} 个")
print(f"  手机银行特征共: {len(MPB_FIELDS)} 个（上表显示实际进入模型的字段）")


## Cell 18：特征重要性（Gain）

In [ ]:
# ── Cell 18：特征重要性（Gain 方法） ──

_importance_models = (
    list(final_model) if isinstance(final_model, (list, tuple)) else [final_model]
)
_normalised_gain_rows = []
for _model in _importance_models:
    _gain = np.asarray(_model.feature_importance(importance_type="gain"), dtype=float)
    _gain_total = float(_gain.sum())
    _normalised_gain_rows.append(_gain / _gain_total if _gain_total > 0 else _gain)
importance_vals = np.mean(np.vstack(_normalised_gain_rows), axis=0)
importance_df   = pd.DataFrame({
    "feature":    feature_names,
    "importance": importance_vals,
}).sort_values("importance", ascending=False).reset_index(drop=True)
importance_df["rank"] = importance_df.index + 1

# Top 20 打印
top20 = importance_df.head(20)
print("===== Top 20 特征重要性（子模型内归一化 Gain 后平均）=====")
print(top20[["rank", "feature", "importance"]].to_string(index=False))

# Top 20 可视化
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20["feature"][::-1], top20["importance"][::-1], color="steelblue")
ax.set_xlabel("平均归一化 Gain")
ax.set_title(f"Top 20 特征重要性（归一化 Gain）| {TARGET}")
plt.tight_layout()
plt.savefig(f"potential_feature_importance_top20_{TARGET}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"图片已保存: potential_feature_importance_top20_{TARGET}.png")

# 科创人才字段 + credamt 专项排名
kechuang_fields = ["当前aum", "当前lum", "科技人才对应得分", "kum分", "lum分", "总分", "aum分", "档位", "credamt"]
print("\n===== 科创人才字段 + credamt 特征重要性 =====")
found, notfound = [], []
for f in kechuang_fields:
    row = importance_df[importance_df["feature"] == f]
    if len(row) > 0:
        found.append({"字段名": f, "Gain": row.iloc[0]["importance"], "排名": row.iloc[0]["rank"]})
    else:
        notfound.append(f)
if found:
    print(pd.DataFrame(found).sort_values("排名").to_string(index=False))
if notfound:
    print(f"  不在特征集中（已删除或数据中不存在）: {notfound}")

print(f"\n{'='*50}")
print(f"              建模汇总")
print(f"{'='*50}")
print(f"  目标变量         : {TARGET}")
print(f"  训练集样本数     : {len(X_train):,}")
print(f"  测试集样本数     : {len(X_test):,}")
print(f"  最终特征数       : {len(feature_names):,}")
print(f"  最优迭代轮次     : {_best_rounds(final_model)}")
print(f"  测试集Top-ρ阈值 : {metrics_82['threshold']:.4f}")
print(f"  AUC              : {metrics_82['auc']:.4f}")
print(f"  AUPRC            : {metrics_82['auprc']:.4f}  (基线={metrics_82['baseline']:.4f}, 提升{metrics_82['auprc']/metrics_82['baseline']:.1f}x)")
print(f"  Accuracy         : {metrics_82['acc']:.4f}")
print(f"  Recall           : {metrics_82['recall']:.4f}")
print(f"  Precision        : {metrics_82['precision']:.4f}")
print(f"  F1-score         : {metrics_82['f1']:.4f}")
print(f"  F2-score         : {metrics_82['f2']:.4f}  (beta=2，更重视Recall)")
print(f"  MCC              : {metrics_82['mcc']:.4f}")
spw_used = params_run.get('scale_pos_weight', '未启用')
print(f"  scale_pos_weight : {spw_used}")
print(f"{'='*50}")

## Cell 20：两份数据客户群体重叠分析

In [ ]:
# ── Cell 20：两份数据客户群体重叠分析 ──
# 依赖：Cell 1 中的文件路径、到期日与起息日筛选配置
# 本 Cell 独立运行，不依赖 Cell 3 的预处理结果

if not FILE_PATH_2:
    print("FILE_PATH_2 为空，跳过重叠分析")
else:
    from load_kechuang_potential_data import (
        read_data, rename_kechuang_cols, filter_by_maturity, filter_by_eff_date,
    )

    def _get_cst_ids(path, apply_maturity_filter, cutoff, apply_eff_date_filter, eff_lower, eff_upper):
        """按主流程同样的日期口径读取，返回非空 cst_id 集合。"""
        df = read_data(path)
        df = rename_kechuang_cols(df)
        if apply_maturity_filter:
            df = filter_by_maturity(df, apply_filter=True, maturity_cutoff=cutoff)
        if apply_eff_date_filter:
            df = filter_by_eff_date(
                df, apply_filter=True, lower=eff_lower, upper=eff_upper,
            )
        if "cst_id" not in df.columns:
            raise ValueError(f"{path} 中未找到 cst_id 列")
        ids = df["cst_id"].astype("string").str.strip()
        ids = ids[ids.notna() & ids.ne("")]
        return set(ids.tolist())

    print(f"正在读取第一份数据: {FILE_PATH}")
    ids_1 = _get_cst_ids(
        FILE_PATH, APPLY_MATURITY_FILTER, MATURITY_CUTOFF,
        APPLY_EFF_DATE_FILTER, EFF_DATE_LOWER, EFF_DATE_UPPER,
    )

    print(f"\n正在读取第二份数据: {FILE_PATH_2}")
    ids_2 = _get_cst_ids(
        FILE_PATH_2, APPLY_MATURITY_FILTER, MATURITY_CUTOFF,
        APPLY_EFF_DATE_FILTER, EFF_DATE_LOWER, EFF_DATE_UPPER,
    )

    overlap     = ids_1 & ids_2
    only_in_1   = ids_1 - ids_2
    only_in_2   = ids_2 - ids_1
    union       = ids_1 | ids_2

    print(f"\n{'━'*52}")
    print(f"  {'数据集':<20s}  {'客户数':>10s}")
    print(f"  {FILE_PATH:<20s}  {len(ids_1):>10,}")
    print(f"  {FILE_PATH_2:<20s}  {len(ids_2):>10,}")
    print(f"{'━'*52}")
    def _safe_ratio(numerator, denominator):
        return numerator / denominator if denominator else float("nan")

    print(f"  两份数据均有（重叠）  : {len(overlap):>8,}  ({_safe_ratio(len(overlap), len(union))*100:.2f}% of 并集)")
    print(f"  仅在第一份中          : {len(only_in_1):>8,}  ({_safe_ratio(len(only_in_1), len(ids_1))*100:.2f}% of 第一份)")
    print(f"  仅在第二份中          : {len(only_in_2):>8,}  ({_safe_ratio(len(only_in_2), len(ids_2))*100:.2f}% of 第二份)")
    print(f"  并集（去重总客户数）  : {len(union):>8,}")
    print(f"{'━'*52}")
    print(f"  Jaccard 相似度        : {_safe_ratio(len(overlap), len(union)):.4f}")
    print(f"  重叠率（相对第一份）  : {_safe_ratio(len(overlap), len(ids_1)):.4f}")
    print(f"  重叠率（相对第二份）  : {_safe_ratio(len(overlap), len(ids_2)):.4f}")
    print(f"{'━'*52}")
